# Multiverse Hybrid v3.0 — Stage 2 PRICE / EV 2000 v1

Stage 1の全券種確率とStage 0の当時締切オッズをexact joinし、事前登録済みのraw EV / market-shape edgeカタログを生成します。

- ワイドは `low` オッズを主判定
- `high` は診断専用
- RESULT / PAYOUT / Settlement / realized ROI は読みません
- 閾値・買い目・Portfolio・資金配分はまだ選びません
- 成果物はGoogle Driveへ保存し、自動ダウンロードは行いません

iPhoneでは **ランタイム → すべてのセルを実行** だけで構いません。


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, shutil, hashlib, json, zipfile

drive.mount('/content/drive')
MY=Path('/content/drive/MyDrive')
REPO=Path('/content/multiverse-research-stage2')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.check_call(['git','clone','--depth','1','https://github.com/fufufu1116/multiverse-research.git',str(REPO)])

EXPECTED={
  'v3/historical_all_market/stage2_price_ev_catalog_engine_v1.py':'9f240c758cb2596e9c67a5214c4e2c610eb82769',
  'v3/historical_all_market/governance/STAGE2_PRICE_EV_SEMANTICS_PREREG_v1.md':'4043a07a88fef45014df2df9df5194514c24faf9',
  'v3/historical_all_market/runtime_receipts/ALL_MARKET_STAGE2_PRICE_EV_STATIC_SELF_CHECK_v1.json':'14edf17dc1f7546a43d4ee72dface0015b7f198d',
}
for rel,exp in EXPECTED.items():
    obs=subprocess.check_output(['git','-C',str(REPO),'hash-object',rel],text=True).strip()
    if obs!=exp: raise RuntimeError(f'FAIL-CLOSED Git blob mismatch {rel}: {obs} != {exp}')
print('✅ STAGE2 EXACT CODE / PREREG / SELF-CHECK BINDINGS PASS')

PRICE=MY/'MULTIVERSE_ALL_MARKET_STAGE0_PRICE_RECOVERY_v2'/'PRICE_ONLY'/'DEV2000_ALL_MARKET_PRICE_CATALOGS_v2.jsonl'
PROB=MY/'MULTIVERSE_ALL_MARKET_STAGE1_PL_PROBABILITY_v1'/'DEV2000_ALL_MARKET_TICKET_PROBABILITIES_PL_v1.jsonl'
OUT=MY/'MULTIVERSE_ALL_MARKET_STAGE2_PRICE_EV_v1'
OUT.mkdir(parents=True,exist_ok=True)
CAT=OUT/'DEV2000_ALL_MARKET_PRICE_EV_CATALOG_v1.jsonl'
QUALITY=OUT/'STAGE2_PRICE_EV_CATALOG_QUALITY_v1.json'
RECEIPT=OUT/'STAGE2_PRICE_EV_RECEIPT_v1.json'
LOG=OUT/'STAGE2_PRICE_EV_RUN_LOG_v1.txt'
ZIP=OUT/'MULTIVERSE_ALL_MARKET_STAGE2_PRICE_EV_v1_ARTIFACT.zip'

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(1<<20),b''): h.update(c)
    return h.hexdigest()

# Existing PASS fast-path.
if RECEIPT.is_file() and QUALITY.is_file() and CAT.is_file():
    r=json.loads(RECEIPT.read_text(encoding='utf-8'))
    q=json.loads(QUALITY.read_text(encoding='utf-8'))
    existing_ok=(
      r.get('status')=='PASS' and q.get('status')=='PASS' and
      r.get('catalog_sha256')==sha256(CAT) and
      r.get('quality_sha256')==sha256(QUALITY) and
      r.get('ticket_join_mismatches')==0 and
      r.get('result_access') is False and
      r.get('settlement_access') is False and
      r.get('realized_roi_computed') is False and
      r.get('threshold_selected') is False and
      r.get('portfolio_constructed') is False and
      r.get('ECON_HOLDOUT1000')=='SEALED'
    )
else:
    existing_ok=False

if existing_ok:
    print('✅ STAGE2 ALREADY PASS — 再計算不要')
    print('Catalog SHA:',sha256(CAT))
    print('Quality SHA:',sha256(QUALITY))
else:
    for p in (CAT,QUALITY,RECEIPT,LOG,ZIP):
        if p.exists(): p.unlink()
    if not PRICE.is_file() or not PROB.is_file():
        raise RuntimeError(f'FAIL-CLOSED input missing PRICE={PRICE.exists()} PROB={PROB.exists()}')
    cmd=['python',str(REPO/'v3/historical_all_market/stage2_price_ev_catalog_engine_v1.py'),str(PRICE),str(PROB),str(CAT),str(QUALITY)]
    print('▶ STAGE2 PRICE/EV 2000R start')
    proc=subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    LOG.write_text(proc.stdout,encoding='utf-8')
    print(proc.stdout)
    if proc.returncode!=0:
        raise RuntimeError(f'FAIL-CLOSED Stage2 return={proc.returncode}; see {LOG}')
    q=json.loads(QUALITY.read_text(encoding='utf-8'))
    if q.get('status')!='PASS' or q.get('ticket_join_mismatches')!=0:
        raise RuntimeError('FAIL-CLOSED Stage2 quality gate')
    if q.get('result_access') is not False or q.get('settlement_access') is not False or q.get('realized_roi_computed') is not False:
        raise RuntimeError('FAIL-CLOSED Stage2 result/settlement firewall')
    if q.get('threshold_selected') is not False or q.get('portfolio_constructed') is not False:
        raise RuntimeError('FAIL-CLOSED Stage2 decision-rule firewall')
    receipt={
      'record':'STAGE2_PRICE_EV_RECEIPT_v1','status':'PASS',
      'stage2_engine_git_blob':'9f240c758cb2596e9c67a5214c4e2c610eb82769',
      'stage2_prereg_git_blob':'4043a07a88fef45014df2df9df5194514c24faf9',
      'price_sha256':sha256(PRICE),'ticket_probability_sha256':sha256(PROB),
      'catalog_sha256':sha256(CAT),'quality_sha256':sha256(QUALITY),
      'races':q['races'],'output_rows':q['output_rows'],
      'ticket_join_mismatches':q['ticket_join_mismatches'],
      'wide_primary_price_rule':q['wide_primary_price_rule'],
      'wide_high_price_role':q['wide_high_price_role'],
      'scientific_trial_count':0,'result_access':False,'settlement_access':False,
      'realized_roi_computed':False,'threshold_selected':False,'portfolio_constructed':False,
      'ECON_HOLDOUT1000':'SEALED'
    }
    RECEIPT.write_text(json.dumps(receipt,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
    with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in (CAT,QUALITY,RECEIPT,LOG): z.write(p,arcname=p.name)
    receipt['artifact_sha256']=sha256(ZIP)
    RECEIPT.write_text(json.dumps(receipt,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')
    print('✅ STAGE2 PASS')
    print(RECEIPT.read_text(encoding='utf-8'))

print('Drive folder:',OUT)
print('RESULT/PAYOUT/Settlement/realized ROI access = none')
print('Threshold/Portfolio selection = none')
print('ECON_HOLDOUT1000 = SEALED')
